In this notebook, we build a baseline model to predict the frames win percentage by player1. In particular, the model predict the player with higher elo rating to be the winner. Then it predicts the winner to have the average win percentage p in the training set and 1-p for the loser. We record the mse on the test set.

In [1]:
import pandas as pd
import numpy as np

In [4]:
#Import the match data for the last 50 tournaments
data = pd.read_csv('../../3_Player_Data_Generation/match_data_300_tourns_modified.csv')
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34521 entries, 0 to 34520
Data columns (total 29 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   player1                   34521 non-null  object 
 1   player2                   34521 non-null  object 
 2   best_of                   34521 non-null  int64  
 3   player1_elo               34521 non-null  int64  
 4   player2_elo               34521 non-null  int64  
 5   elo_match_win_rate        34521 non-null  float64
 6   elo_frame_win_rate        34521 non-null  float64
 7   p1_matches_played         34521 non-null  int64  
 8   p1_matches_won            34521 non-null  int64  
 9   p1_frames_played          34521 non-null  int64  
 10  p1_frames_won             34521 non-null  int64  
 11  p2_matches_played         34521 non-null  int64  
 12  p2_matches_won            34521 non-null  int64  
 13  p2_frames_played          34521 non-null  int64  
 14  p2_fra

In [5]:
data.head()

,player1,player2,best_of,player1_elo,player2_elo,elo_match_win_rate,elo_frame_win_rate,p1_matches_played,p1_matches_won,p1_frames_played,...,p2_frames_played_1_year,p2_frames_won_1_year,p2_frames_played_3_years,p2_frames_won_3_years,score1,score2,match_result,win_percentage,tournament_id,date
0,Mark Allen,Ricky Walden,7,1580,1418,0.709992,0.599888,501,313,3345,...,45,25,1032,557,0,4,1.0,0.000000,1063,1.415318e+18
1,Stephen Maguire,Judd Trump,7,1563,1551,0.516401,0.507499,663,444,4995,...,122,79,1344,782,1,4,1.0,0.200000,1063,1.415059e+18
2,Mark Selby,Steve Davis,7,1592,1246,0.878716,0.703704,736,495,5330,...,8,2,581,286,4,1,0.0,0.800000,1063,1.415059e+18
3,Neil Robertson,Ali Carter,7,1546,1544,0.502734,0.501250,621,402,4581,...,11,5,876,499,4,0,0.0,1.000000,1063,1.415318e+18
4,Stuart Bingham,Ronnie O'Sullivan,7,1488,1663,0.275106,0.392337,744,462,5592,...,71,46,674,431,2,4,1.0,0.333333,1063,1.415146e+18


In [6]:
#Train test split
from sklearn.model_selection import train_test_split
data_train, data_test = train_test_split(data, 
                                        test_size = 0.2)

In [7]:
# find the average win percentage of winner from training set.
winner_win_percs = []
for i in range(len(data_train)):
    match = data_train.iloc[i]
    match_result = match['match_result']
    win_perc = match['win_percentage']
    if match_result:
        winner_win_percs.append(1-win_perc)
    else: 
        winner_win_percs.append(win_perc)

print(winner_win_percs[:5])


[np.float64(0.75), np.float64(1.0), np.float64(0.75), np.float64(0.5454545454545454), np.float64(1.0)]


In [8]:
average_winner_win_perc = np.mean(winner_win_percs)
average_winner_win_perc


np.float64(0.7531679501405557)

In [9]:
#Making prediction on test set.
win_per_prediction = np.zeros(len(data_test))
for i in range(len(data_test)):
    match = data_test.iloc[i]
    if match['elo_match_win_rate']>=0.5:
        win_per_prediction[i] = average_winner_win_perc
    else: 
        win_per_prediction[i] = 1 - average_winner_win_perc

print(win_per_prediction[:5])


[0.24683205 0.24683205 0.75316795 0.75316795 0.24683205]


In [10]:
#Calculate rmse
from sklearn.metrics import mean_absolute_error
score = mean_absolute_error(win_per_prediction, data_test['win_percentage'].values)
print(score)

0.24552377709653203
